# Physics-Informed Neural Networks for Activity Coefficient Prediction

This notebook presents the development of a physics-informed neural network (PINN) to predict activity coefficients using processed experimental data. By embedding thermodynamic constraints directly into the loss function, we incorporate domain knowledge into the model, ensuring thermodynamic consistency while leveraging a data-driven approach.

## Dependencies

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from pathlib import Path
from sklearn.model_selection import KFold, train_test_split
from sklearn.preprocessing import MinMaxScaler

## PINN modeling

### Designing the Neural Network Components

Define the architecture of the neural network

In [2]:
torch.manual_seed(42)


class NeuralNetArchitecture(nn.Module):
    def __init__(self, in_dim: int, out_dim: int) -> None:
        super().__init__()
        self.layer_1 = nn.Linear(in_dim, 64)
        self.layer_2 = nn.Linear(64, 32)
        self.layer_3 = nn.Linear(32, 16)
        self.layer_4 = nn.Linear(16, out_dim)

    def forward(self, x):
        x = F.relu(self.layer_1(x))
        x = F.relu(self.layer_2(x))
        x = F.relu(self.layer_3(x))
        x = self.layer_4(x)

        return x

Implement a custom loss function that enforces thermodynamic consistency by embedding the Gibbs-Duhem equation into the standard MSE loss

In [3]:
def gibbs_duhem_loss(
    y_pred: torch.Tensor, y_true: torch.Tensor, X: torch.Tensor, lambda_gd: int = 1
) -> torch.Tensor:
    # MSE error
    mse_loss = torch.mean((y_pred - y_true) ** 2)

    # Gibbs Duhem loss
    gamma1 = y_pred[:, 0]
    dgamma1_dx1 = torch.autograd.grad(
        gamma1,
        X,
        grad_outputs=torch.ones_like(gamma1),
        create_graph=True,
        allow_unused=True,
    )[0][:, 3]

    gamma2 = y_pred[:, 1]
    dgamma2_dx2 = torch.autograd.grad(
        gamma2,
        X,
        grad_outputs=torch.ones_like(gamma2),
        create_graph=True,
        allow_unused=True,
    )[0][:, 6]

    gibbs_duhem_loss = torch.mean((dgamma1_dx1 + dgamma2_dx2) ** 2)

    # Total loss
    loss = mse_loss + gibbs_duhem_loss * lambda_gd
    return loss

### Reading the data

Defining the data path conveniently:

In [4]:
DATA_PATH = Path("../data")

Reading the data into a `pandas.DataFrame`:

In [5]:
data = pd.read_csv(DATA_PATH / "processed" / "toy_problem_input_dataset.csv")

data

,r_1,q_1,x_1,r_2,q_2,x_2,T,gamma_1,gamma_2
0,9.5482,7.976,0.00,5.8486,4.936,1.00,398.839,1.10848,1.00000
1,9.5482,7.976,0.01,5.8486,4.936,0.99,399.213,1.10353,1.00002
2,9.5482,7.976,0.02,5.8486,4.936,0.98,399.590,1.09880,1.00008
3,9.5482,7.976,0.03,5.8486,4.936,0.97,399.971,1.09428,1.00018
4,9.5482,7.976,0.04,5.8486,4.936,0.96,400.355,1.08995,1.00032
...,...,...,...,...,...,...,...,...,...
4035,13.5946,11.216,0.96,3.9243,3.668,0.04,509.434,1.64304,1.00037
4036,13.5946,11.216,0.97,3.9243,3.668,0.03,532.267,1.57456,1.00018
4037,13.5946,11.216,0.98,3.9243,3.668,0.02,568.855,1.47692,1.00006
4038,13.5946,11.216,0.99,3.9243,3.668,0.01,600.617,1.40596,1.00001


Remove rows with NaN values:

In [6]:
data.dropna(inplace=True)
data.reset_index(drop=True, inplace=True)

data

,r_1,q_1,x_1,r_2,q_2,x_2,T,gamma_1,gamma_2
0,9.5482,7.976,0.00,5.8486,4.936,1.00,398.839,1.10848,1.00000
1,9.5482,7.976,0.01,5.8486,4.936,0.99,399.213,1.10353,1.00002
2,9.5482,7.976,0.02,5.8486,4.936,0.98,399.590,1.09880,1.00008
3,9.5482,7.976,0.03,5.8486,4.936,0.97,399.971,1.09428,1.00018
4,9.5482,7.976,0.04,5.8486,4.936,0.96,400.355,1.08995,1.00032
...,...,...,...,...,...,...,...,...,...
4031,13.5946,11.216,0.96,3.9243,3.668,0.04,509.434,1.64304,1.00037
4032,13.5946,11.216,0.97,3.9243,3.668,0.03,532.267,1.57456,1.00018
4033,13.5946,11.216,0.98,3.9243,3.668,0.02,568.855,1.47692,1.00006
4034,13.5946,11.216,0.99,3.9243,3.668,0.01,600.617,1.40596,1.00001


### PINN training

Defining the device and splitting the data into training and test sets

In [7]:
# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

features = data.iloc[:, :-2].values
labels = data.iloc[:, -2:].values

X_train, X_test, y_train, y_test = train_test_split(
    features, labels, test_size=0.1, random_state=42
)

Defining the cross-validation setup

In [8]:
k_folds = 10
kf = KFold(n_splits=k_folds, shuffle=True, random_state=42)

Specifying training hyperparameters

In [9]:
num_epochs = 10000
learning_rate = 0.001
lambda_gd = 1e-1
patience = 1000

Executing the training loop for the PINN

In [10]:
fold_losses = []
val_losses = []
test_losses = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X_train)):
    print(f"Fold {fold + 1}/{k_folds}")

    # Train-validation split for this fold
    X_train_cv, X_val = X_train[train_idx], X_train[val_idx]
    y_train_cv, y_val = y_train[train_idx], y_train[val_idx]

    # Scale features
    scaler = MinMaxScaler()
    X_train_scaled = scaler.fit_transform(X_train_cv)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)

    # Convert to tensors and move to device
    X_train_tensor = torch.tensor(
        X_train_scaled, dtype=torch.float32, requires_grad=True
    ).to(device)
    X_val_tensor = torch.tensor(
        X_val_scaled, dtype=torch.float32, requires_grad=True
    ).to(device)
    X_test_tensor = torch.tensor(
        X_test_scaled, dtype=torch.float32, requires_grad=True
    ).to(device)
    y_train_tensor = torch.tensor(y_train_cv, dtype=torch.float32).to(device)
    y_val_tensor = torch.tensor(y_val, dtype=torch.float32).to(device)
    y_test_tensor = torch.tensor(y_test, dtype=torch.float32).to(device)

    # Model and optimizer
    input_size = X_train_cv.shape[1]
    output_size = 2
    model = NeuralNetArchitecture(input_size, output_size).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    # Early stopping vars
    best_train_loss = float("inf")
    best_val_loss = float("inf")
    best_model = None
    epochs_no_improve = 0

    # Training loop
    for epoch in range(num_epochs):
        model.train()

        # Forward pass
        outputs = model(X_train_tensor)

        # Loss
        loss = gibbs_duhem_loss(outputs, y_train_tensor, X_train_tensor, lambda_gd)

        # Validation loss
        val_outputs = model(X_val_tensor)
        val_loss = gibbs_duhem_loss(val_outputs, y_val_tensor, X_val_tensor, lambda_gd)

        # Backward propagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Early stopping check
        if val_loss < best_val_loss:
            best_train_loss = loss
            best_val_loss = val_loss
            best_model = model
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= patience:
            print(f"Early stopping at epoch {epoch + 1}")
            break

        if (epoch + 1) % 20 == 0:
            print(f"[Fold {fold + 1} | Epoch {epoch + 1}]: Training loss -> {loss}")
            print(
                f"[Fold {fold + 1} | Epoch {epoch + 1}]: Validation loss -> {val_loss}"
            )

    # Evaluate test data
    test_outputs = best_model(X_test_tensor)
    test_loss = gibbs_duhem_loss(test_outputs, y_test_tensor, X_test_tensor, lambda_gd)

    # Store best loss for this fold
    fold_losses.append(best_train_loss.item())
    val_losses.append(best_val_loss.item())
    test_losses.append(test_loss.item())

# Print average loss across folds
print(f"Average loss across {k_folds} folds: {np.mean(fold_losses)}")
print(f"Average validation loss across {k_folds} folds: {np.mean(val_losses)}")
print(f"Average test loss across {k_folds} folds: {np.mean(test_losses)}")

Fold 1/10
[Fold 1 | Epoch 20]: Training loss -> 7.341529369354248
[Fold 1 | Epoch 20]: Validation loss -> 6.154910564422607
[Fold 1 | Epoch 40]: Training loss -> 6.7735419273376465
[Fold 1 | Epoch 40]: Validation loss -> 5.614776134490967
[Fold 1 | Epoch 60]: Training loss -> 5.104795455932617
[Fold 1 | Epoch 60]: Validation loss -> 4.053679943084717
[Fold 1 | Epoch 80]: Training loss -> 4.156768798828125
[Fold 1 | Epoch 80]: Validation loss -> 3.310368537902832
[Fold 1 | Epoch 100]: Training loss -> 3.8639817237854004
[Fold 1 | Epoch 100]: Validation loss -> 2.996962308883667
[Fold 1 | Epoch 120]: Training loss -> 3.6446428298950195
[Fold 1 | Epoch 120]: Validation loss -> 2.8017778396606445
[Fold 1 | Epoch 140]: Training loss -> 3.4743762016296387
[Fold 1 | Epoch 140]: Validation loss -> 2.6499648094177246
[Fold 1 | Epoch 160]: Training loss -> 3.35276198387146
[Fold 1 | Epoch 160]: Validation loss -> 2.545811176300049
[Fold 1 | Epoch 180]: Training loss -> 3.266787052154541
[Fold 1 